# Pipeline de anotação de enquadramento jornalístico

## 1. Dependências e caminhos

In [ ]:
!pip install -q litellm pydantic

from pathlib import Path

def achar_raiz() -> Path:
    """Sobe a partir do diretório atual até encontrar noticias/ e prompts/."""
    atual = Path.cwd().resolve()
    for base in [atual, *atual.parents]:
        if (base / "noticias").is_dir() and (base / "prompts").is_dir():
            return base
    raise FileNotFoundError(f"noticias/ e prompts/ não encontrados a partir de {atual}")

RAIZ = achar_raiz()
DIR_NOTICIAS = RAIZ / "noticias"
DIR_PROMPTS = RAIZ / "prompts"
DIR_RESULTADOS_V3 = RAIZ / "resultados" / "v3"
DIR_SAIDA = Path.cwd() / "saida"

print("Raiz do repositorio:", RAIZ)
print(sorted(p.name for p in DIR_NOTICIAS.glob("*.json")))


## 2. Configurar anotadores, categorias e parâmetros

In [ ]:
import json, os, re, time, unicodedata
from typing import Literal

import litellm
from pydantic import BaseModel, Field, ValidationError

# chave curta (usada nos nomes dos arquivos) -> identificador no OpenRouter
MODELOS = {
    "deepseek-v4-pro": "openrouter/deepseek/deepseek-v4-pro",
    "gpt-5.4": "openrouter/openai/gpt-5.4",
    "claude-sonnet-4.6": "openrouter/anthropic/claude-sonnet-4.6",
}

CATEGORIAS = {
    "S1": "Marcação lexical avaliativa",
    "S2": "Enquadramento causal-moral",
    "S3": "Suporte evidencial e atribuição",
    "S4": "Pluralidade e contraponto interno",
    "S5": "Contextualização episódica",
}

TEMPERATURA, TOP_P, MAX_TOKENS = 0.1, 0.9, 4096

## 3. Definir os schemas

In [ ]:
class Noticia(BaseModel):
    id: str
    veiculo: str
    data: str
    titulo: str
    texto: str
    categoria: str = ""


class SubSinal(BaseModel):
    """Um subcritério detectado, sempre preso a um trecho literal."""
    sub_sinal: str
    trecho: str = Field(min_length=1)
    justificativa: str = Field(min_length=1)
    confianca: Literal["alta", "media", "baixa"]
    limitacao: str = ""


class Sinal(BaseModel):
    """Uma das 5 categorias: presente com subcritérios, ou ausente com justificativa."""
    sinal: Literal["S1", "S2", "S3", "S4", "S5"]
    presente: bool
    sub_sinais_detectados: list[SubSinal] = []
    justificativa_ausencia: str = ""


class Analise(BaseModel):
    raciocinio: str = ""
    sinais: list[Sinal]
    resumo: str = ""

## 4. Carregar a notícia

In [ ]:
def carregar_noticia(noticia_id: str) -> Noticia:
    dados = json.loads((DIR_NOTICIAS / f"{noticia_id}.json").read_text(encoding="utf-8"))
    return Noticia.model_validate(dados)

def listar_noticias() -> list:
    return sorted(p.stem for p in DIR_NOTICIAS.glob("N*.json"))

n01 = carregar_noticia("N01")
print(f"{n01.id} | {n01.veiculo} | {n01.data}")
print(n01.titulo)
print()
print(n01.texto[:500], "...")

## 5. Carregar o protocolo

In [ ]:
def carregar_protocolo(versao: str = "v3") -> str:
    nome = "v0-sem-protocolo.txt" if versao == "v0" else f"{versao}.txt"
    return (DIR_PROMPTS / nome).read_text(encoding="utf-8")

protocolo = carregar_protocolo("v3")
print(f"{len(protocolo)} caracteres\n")
print(protocolo[:1200], "...")

## 6. Montar o prompt

In [ ]:
def montar_mensagens(protocolo: str, noticia: Noticia) -> list:
    mensagem_usuario = (
        "TEXTO PARA ANÁLISE:\n\n"
        f"Veículo: {noticia.veiculo}\n"
        f"Data: {noticia.data}\n"
        f"Título: {noticia.titulo}\n\n"
        "---\n\n"
        f"{noticia.texto}"
    )
    return [
        {"role": "system", "content": protocolo},
        {"role": "user", "content": mensagem_usuario},
    ]

msgs = montar_mensagens(protocolo, n01)
print(f"system: {len(msgs[0]['content'])} chars | user: {len(msgs[1]['content'])} chars")
print(msgs[1]["content"][:300], "...")

## 7. Anotar

In [ ]:
def _extrair_json(bruto: str) -> dict:
    """Modelos às vezes envolvem o JSON em ```json ... ```."""
    texto = bruto.strip()
    if texto.startswith("```"):
        linhas = texto.split("\n")
        fim = len(linhas) - 1 if linhas[-1].strip() == "```" else len(linhas)
        texto = "\n".join(linhas[1:fim]).strip()
    return json.loads(texto)


def anotar_online(mensagens: list, modelo_openrouter: str) -> dict:
    chave = os.getenv("OPENROUTER_API_KEY")
    if not chave:
        raise RuntimeError("OPENROUTER_API_KEY não encontrada — use a rota offline.")

    inicio, erro, analise, uso = time.time(), None, None, None
    try:
        resposta = litellm.completion(
            model=modelo_openrouter,
            messages=mensagens,
            temperature=TEMPERATURA,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            api_key=chave,
        )
        uso = resposta.usage
        analise = Analise.model_validate(
            _extrair_json(resposta.choices[0].message.content or "")
        )
    except json.JSONDecodeError as e:
        erro = f"JSON inválido: {e}"
    except ValidationError as e:
        erro = f"Fora do schema: {e.error_count()} erro(s)"
    except Exception as e:
        erro = f"{type(e).__name__}: {e}"

    return {
        "noticia_id": None,
        "modelo": modelo_openrouter,
        "tokens_prompt": getattr(uso, "prompt_tokens", 0),
        "tokens_resposta": getattr(uso, "completion_tokens", 0),
        "tempo_segundos": round(time.time() - inicio, 2),
        "erro": erro,
        "analise": analise,
    }


def anotar_offline(noticia_id: str, modelo_curto: str) -> dict:
    dados = json.loads(
        (DIR_RESULTADOS_V3 / f"{noticia_id}-{modelo_curto}.json").read_text(encoding="utf-8"))
    if isinstance(dados, list):
        dados = dados[0]
    try:
        dados["analise"] = Analise.model_validate(dados["analise"])
    except ValidationError as e:
        dados["analise"] = None
        dados["erro"] = f"Fora do schema: {e.error_count()} erro(s)"
    return dados

exemplo = anotar_offline("N01", "gpt-5.4")
print(exemplo["modelo"], "|", exemplo["tokens_prompt"], "+", exemplo["tokens_resposta"], "tokens")
print(exemplo["analise"].raciocinio[:400], "...")

## 8. Validar a saída

In [ ]:
_ASPAS = str.maketrans({
    "“": '"', "”": '"', "„": '"', "«": '"', "»": '"',
    "‘": "'", "’": "'", "–": "-", "—": "-", "…": "...",
})

def _normalizar(texto: str) -> str:
    texto = texto.translate(_ASPAS)
    sem_acento = unicodedata.normalize("NFKD", texto)
    sem_acento = "".join(c for c in sem_acento if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", sem_acento).strip().lower()


def validar(analise: Analise, noticia: Noticia) -> dict:
    problemas = []

    codigos = [s.sinal for s in analise.sinais]
    if codigos != list(CATEGORIAS):
        problemas.append(f"categorias esperadas {list(CATEGORIAS)}, obtidas {codigos}")

    texto_normalizado = _normalizar(noticia.texto + " " + noticia.titulo)
    total = ancorados = 0

    for sinal in analise.sinais:
        for sub in sinal.sub_sinais_detectados:
            total += 1
            if _normalizar(sub.trecho) in texto_normalizado:
                ancorados += 1
            else:
                problemas.append(
                    f"{sinal.sinal}/{sub.sub_sinal}: trecho não encontrado "
                    f'literalmente — "{sub.trecho[:60]}..."')
        if sinal.presente and not sinal.sub_sinais_detectados:
            problemas.append(f"{sinal.sinal}: presente=true sem subcritério")

    return {"valido": not problemas, "trechos": total, "ancorados": ancorados,
            "problemas": problemas}


def presencas(analise: Analise) -> dict:
    return {s.sinal: s.presente for s in analise.sinais}

v = validar(exemplo["analise"], n01)
print(v["ancorados"], "/", v["trechos"], "trechos ancorados |", presencas(exemplo["analise"]))
for p in v["problemas"]:
    print("  -", p)

## 9. Medir a concordância

In [ ]:
def fleiss_kappa(itens: list, n_anotadores: int) -> float:
    """`itens` = quantos anotadores marcaram 'presente' em cada item."""
    n_itens = len(itens)
    if n_itens == 0:
        return float("nan")

    # concordância observada: proporção de pares de anotadores que concordam
    soma = 0.0
    for presentes in itens:
        ausentes = n_anotadores - presentes
        soma += (presentes**2 + ausentes**2 - n_anotadores) / (n_anotadores * (n_anotadores - 1))
    p_obs = soma / n_itens

    # concordância esperada por acaso, a partir das marginais
    proporcao = sum(itens) / (n_itens * n_anotadores)
    p_esp = proporcao**2 + (1 - proporcao) ** 2

    return float("nan") if p_esp == 1 else (p_obs - p_esp) / (1 - p_esp)


def calcular_concordancia(matriz: dict, noticias: list, modelos: list) -> dict:
    n_anot = len(modelos)

    def contagens(cats):
        return [sum(matriz[(n, m)][c] for m in modelos) for n in noticias for c in cats]

    glob = contagens(list(CATEGORIAS))
    r = {
        "n_itens": len(glob),
        "kappa_global": fleiss_kappa(glob, n_anot),
        "kappa_por_categoria": {c: fleiss_kappa(contagens([c]), n_anot) for c in CATEGORIAS},
        "unanimidade_presente": sum(1 for c in glob if c == n_anot),
        "unanimidade_ausente": sum(1 for c in glob if c == 0),
    }
    r["divergentes"] = r["n_itens"] - r["unanimidade_presente"] - r["unanimidade_ausente"]
    return r

## 10. Rodar

In [ ]:
def executar(noticias_ids=None, modelos=None, online=False, versao_protocolo="v3", verboso=False):
    noticias_ids = noticias_ids or listar_noticias()
    modelos = modelos or list(MODELOS)
    print(f"Modo: {'ONLINE' if online else 'OFFLINE'} | {len(noticias_ids)} notícias | "
          f"{len(modelos)} modelos | protocolo {versao_protocolo}\n")

    protocolo = carregar_protocolo(versao_protocolo)
    matriz, analises_por_noticia, validacoes = {}, {}, []

    for noticia_id in noticias_ids:
        noticia = carregar_noticia(noticia_id)
        analises_por_noticia[noticia_id] = {}

        for modelo in modelos:
            if online:
                relatorio = anotar_online(montar_mensagens(protocolo, noticia), MODELOS[modelo])
                relatorio["noticia_id"] = noticia_id
            else:
                relatorio = anotar_offline(noticia_id, modelo)

            if relatorio.get("erro"):
                print(f"  {noticia_id} | {modelo}: ERRO — {relatorio['erro']}")
                continue

            analise = relatorio["analise"]
            validacao = validar(analise, noticia)
            validacao.update({"noticia": noticia_id, "modelo": modelo})
            validacoes.append(validacao)

            marcas = presencas(analise)
            matriz[(noticia_id, modelo)] = marcas
            analises_por_noticia[noticia_id][modelo] = analise

            selo = "ok " if validacao["valido"] else "!! "
            resumo = " ".join(f"{c}{'+' if marcas.get(c) else '-'}" for c in CATEGORIAS)
            print(f"  {selo}{noticia_id} | {modelo:<20} {resumo}  "
                  f"({validacao['ancorados']}/{validacao['trechos']} trechos ancorados)")
            if verboso:
                for p in validacao["problemas"]:
                    print("        -", p)

    completas = [n for n in noticias_ids if all((n, m) in matriz for m in modelos)]
    concordancia = None
    if len(modelos) >= 2 and completas:
        concordancia = calcular_concordancia(matriz, completas, modelos)
        print("\n--- Concordância entre anotadores ---")
        print(f"Itens: {concordancia['n_itens']} ({len(completas)} notícias × {len(CATEGORIAS)} categorias)")
        print(f"Fleiss' kappa global = {concordancia['kappa_global']:.3f}")
        for c, k in concordancia["kappa_por_categoria"].items():
            print(f"  {c} ({CATEGORIAS[c]:<34}) = {k:6.3f}")
        print(f"Unânimes presente: {concordancia['unanimidade_presente']} | "
              f"unânimes ausente: {concordancia['unanimidade_ausente']} | "
              f"divergentes: {concordancia['divergentes']}")

    total = sum(v["trechos"] for v in validacoes)
    ancorados = sum(v["ancorados"] for v in validacoes)
    if total:
        print(f"\nAncoragem literal: {ancorados}/{total} trechos ({ancorados/total*100:.1f}%)")

    return {"analises": analises_por_noticia, "validacoes": validacoes,
            "concordancia": concordancia}

resultado = executar()

## 11. Salvar a saída

In [ ]:
def resumo_legivel(noticia: Noticia, analises: dict) -> str:
    linhas = [f"NOTÍCIA {noticia.id} — {noticia.veiculo} ({noticia.data})",
              f"  {noticia.titulo}", ""]
    for modelo, analise in analises.items():
        marcas = presencas(analise)
        linhas.append(f"  [{modelo}] " + " ".join(
            f"{c}{'+' if marcas.get(c) else '-'}" for c in CATEGORIAS))
        for sinal in analise.sinais:
            for sub in sinal.sub_sinais_detectados:
                linhas.append(f'      {sub.sub_sinal} ({sub.confianca}): "{sub.trecho[:90]}"')
    return "\n".join(linhas)


DIR_SAIDA.mkdir(parents=True, exist_ok=True)

serializavel = {
    "analises": {nid: {m: a.model_dump(mode="json") for m, a in analises.items()}
                 for nid, analises in resultado["analises"].items()},
    "validacoes": resultado["validacoes"],
    "concordancia": resultado["concordancia"],
}
(DIR_SAIDA / "resultado.json").write_text(
    json.dumps(serializavel, ensure_ascii=False, indent=2), encoding="utf-8")

blocos = [resumo_legivel(carregar_noticia(nid), analises)
          for nid, analises in resultado["analises"].items() if analises]
(DIR_SAIDA / "resumo.txt").write_text("\n\n".join(blocos), encoding="utf-8")

print("Salvo em:", DIR_SAIDA)
print((DIR_SAIDA / "resumo.txt").read_text(encoding="utf-8")[:900])

## 12. Variações

In [ ]:
executar(noticias_ids=["N07"], verboso=True)

# executar(modelos=["gpt-5.4", "claude-sonnet-4.6"])
# executar(noticias_ids=["N01"], online=True)
# executar(noticias_ids=["N01"], versao_protocolo="v0", online=True)